## 블루윙스 스마트 팩토리 -- 라인 별 3-클래스 베이스라인 모델 (순정 버전)

[기본 설계] (순정/노튜닝)
- 전처리는 이미 완료된 제품별 파일(A_31, TO_31) 사용, 추가 전처리 없음.
- 전처리 = 제품코드 단위 | 모델링 = 라인 단위 수행 (csv 파일 내 LINE_ 원핫 컬럼 -> 실제 라인 복원 후 분리)
- 후보 모델 3종 (XGB, CAT, LGBM) 모두 사용, 모델 테이블로 비교
- **하이퍼파라미터 튜닝 없음** — 각 라이브러리의 기본값 그대로 사용
- **조기종료 / 내부검증 분리 없음** — 과적합 방어용 추가 장치를 걷어낸 순정 구조

[리키지 방지 — 튜닝 여부와 무관하게 반드시 유지하는 최소 구조]
1) Y_Quality는 타겟으로만 사용, 피쳐에서 제외
2) Product_id, product_code, timestamp, line_원핫 피처 제외
3) MI 기반 피처 선택은 각 fold의 train 구간에서만 계산 -> test에 적용
4) 회귀->분류 임계값도 각 fold의 train에서만 계산
5) 데이터 합성 없이 표본 가중치만 사용 -> 합성 leakage 차단

[타겟 설계]
- Y_Quality를 회귀로 예측 -> 학습 fold의 임계값으로 3-class 분류
- 타깃 표준화(z-score)는 회귀가 정상적으로 분할을 학습하기 위한 최소 조건이라 유지
  (표준화 없이 좁은 범위의 Y_Quality를 그대로 회귀하면 트리가 갈라지지 않는 문제가 있었음)

[순정 버전에서 제거한 것 — 이전 버전과의 차이]
- 하이퍼파라미터(max_depth, reg_alpha/lambda, subsample 등 세부 조정) → 모델 기본값
- inner train/val 분리 + early stopping → 제거, outer fold에서 바로 학습
- k_feat 하한선(10) → 최소 안전장치(5)로 단순화

이 버전은 "아무 것도 튜닝하지 않았을 때 실제로 얼마나 과적합되는가"를 보여주는 기준점이다.
이후 정규화·조기종료 등을 추가하며 얼마나 개선되는지 이 결과와 비교하면 된다.

### 라이브러리 불러오기

In [26]:
# 기본 라이브러리
import pandas as pd
import numpy as np

# 시각화
import seaborn as sns
import matplotlib.pyplot as plt
import koreanize_matplotlib

# 피쳐 선별
from sklearn.feature_selection import mutual_info_regression

# 모델링
import lightgbm as lgb
import xgboost as xgb
import catboost as cat

# 모델 평가
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, recall_score, precision_score, classification_report, confusion_matrix, roc_auc_score, roc_curve, auc

# 경고 무시
import warnings
warnings.filterwarnings('ignore')

# 시드 설정
np.random.seed(42)

### 데이터 불러오기

In [27]:
# 라인 복원
def reconstruct_line(df, dummy_cols, baseline_name):
    line = pd.Series(baseline_name, index=df.index)
    for c in dummy_cols:
        line[df[c]==1] = c.replace('LINE_','')
    return line

LINE_MAP = {
    'A_31_preprocessed.csv': (['LINE_T010306', 'LINE_T050304', 'LINE_T050307'], 'T010305'),
    'TO_31_preprocessed.csv': (['LINE_T100306'], 'T100304'),
}

# 모델링에서 drop할 피처들 (리키지 방지 — 튜닝 여부와 무관하게 항상 유지)
DROP_BASE = ['PRODUCT_ID','Y_Class','Y_Quality','PRODUCT_CODE','TIMESTAMP']

# 데이터 불러오기
def load_data():
    frames = []
    for path, (dummies, baseline) in LINE_MAP.items():
        df = pd.read_csv(path)
        df['LINE'] = reconstruct_line(df, dummies, baseline)
        drop_cols = DROP_BASE + dummies + ['LINE']
        feats = [c for c in df.columns if c not in drop_cols]
        for ln in df['LINE'].unique():
            sub = df[df['LINE']==ln].reset_index(drop=True)
            frames.append((ln, sub, feats))
    return frames

### 후보 모델 3종 (순정 — 라이브러리 기본값)</br>

- Y_Quality 예측 후 고정 임계값으로 등급 분류
- **튜닝 없음**: max_depth, reg_alpha/lambda, subsample 등 일체 미지정 → 각 라이브러리 기본 설정 그대로
- 조기종료도 사용하지 않음 (기본 n_estimators만큼 고정 학습)

In [28]:
def build_model():
    return {
        'LightGBM': lgb.LGBMRegressor(random_state=42, verbose=-1),
        'XGBoost': xgb.XGBRegressor(random_state=42),
        'CatBoost': cat.CatBoostRegressor(random_seed=42, verbose=0, allow_writing_files=False),
    }

타깃 표준화만은 순정 버전에서도 유지한다.
Y_Quality는 값의 범위가 극히 좁아(0.50~0.58, std 매우 작음), 표준화 없이 그대로 회귀하면
트리 기반 모델이 분할 자체를 만들지 못하고 상수(평균)만 예측하는 문제가 있었다.
이는 '과적합 방어'가 아니라 '회귀가 정상적으로 작동하기 위한 최소 조건'이라 순정 버전에도 남긴다.

### 클래스 가중치 </br>
- 표본 가중치로 변환 (참고 설계에서 명시적으로 요구된 항목 — 유지)
- 데이터 합성 없음

In [29]:
def class_sample_weight(y_class):
    vc = pd.Series(y_class).value_counts()
    w_map = {c: len(y_class)/(len(vc) * n) for c, n in vc.items()}
    return np.array([w_map[c] for c in y_class])

### 임계값 계산</br>
- train fold 내에서만 계산 (누수 차단 — 튜닝 여부와 무관하게 항상 유지)

In [30]:
def fit_thresholds(y_quality_tr, y_class_tr):
    q0 = y_quality_tr[y_class_tr == 0]
    q1 = y_quality_tr[y_class_tr == 1]
    q2 = y_quality_tr[y_class_tr == 2]
    thr01 = (q0.max() + q1.min()) / 2 if len(q0) and len(q1) else np.median(y_quality_tr)
    thr12 = (q1.max() + q2.min()) / 2 if len(q1) and len(q2) else np.median(y_quality_tr)
    thr01, thr12 = sorted([thr01, thr12])
    return thr01, thr12

def to_class(pred_quality, thr01, thr12):
    return np.where(pred_quality < thr01, 0, np.where(pred_quality < thr12, 1, 2))

### 라인 x 모델 평가 (순정)</br>
- Stratified K-Fold (outer만 사용, inner 검증 분리 없음)
- Fold 내부 MI 선택 — 리키지 방지 필수 요소라 유지, 단 k_feat 하한을 5로 단순화
- 조기종료 없이 기본 n_estimators로 고정 학습 → train/test Gap이 이전보다 커질 수 있음

In [31]:
# def eval_line_model(sub, feats, model_name, model_builder, n_splits):
#     X = sub[feats]
#     yq = sub['Y_Quality'].values
#     yc = sub['Y_Class'].values
#     n = len(sub)
#     skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
#     oof_pred = np.full(n, -1)
#     tr_f1s, te_f1s = [], []

#     for tr, te in skf.split(X, yc):
#         # 피처 개수: 최소 안전장치만(5), 나머지는 표본 비례 (하이퍼파라미터 튜닝 아님, 구조적 필수)
#         k_feat = 60

#         # 피처 선택 — train에서만 (누수 차단)
#         X_tr_imp = X.iloc[tr].fillna(X.iloc[tr].median()).fillna(0)
#         mi = mutual_info_regression(X_tr_imp, yq[tr], random_state=42)
#         top = X.columns[np.argsort(mi)[::-1][:k_feat]]
#         Xtr, Xte = X.iloc[tr][top], X.iloc[te][top]

#         # 클래스 가중치 — train에서만
#         w_tr = class_sample_weight(yc[tr])

#         # 임계값 — train에서만
#         thr01, thr12 = fit_thresholds(yq[tr], yc[tr])

#         # 타깃 표준화 — train에서만
#         mu, sd = yq[tr].mean(), yq[tr].std()
#         sd = sd if sd > 1e-8 else 1.0
#         yq_tr_s = (yq[tr] - mu) / sd

#         # 회귀 학습 (순정: 조기종료 없이 기본 파라미터로 그냥 학습)
#         model = model_builder()
#         model.fit(Xtr, yq_tr_s, sample_weight=w_tr)

#         tr_pred_c = to_class(model.predict(Xtr) * sd + mu, thr01, thr12)
#         te_pred_c = to_class(model.predict(Xte) * sd + mu, thr01, thr12)
#         oof_pred[te] = te_pred_c
#         tr_f1s.append(f1_score(yc[tr], tr_pred_c, average='macro'))
#         te_f1s.append(f1_score(yc[te], te_pred_c, average='macro'))

#     macro_f1 = f1_score(yc, oof_pred, average='macro')
#     macro_rec = recall_score(yc, oof_pred, average='macro', zero_division=0)
#     macro_pre = precision_score(yc, oof_pred, average='macro', zero_division=0)
#     gap = np.mean(tr_f1s) - np.mean(te_f1s)

#     return {'n': n, 'p_selected': k_feat, 'p/n': round(k_feat / len(tr), 2),
#             'MacroF1': round(macro_f1, 3), 'Recall': round(macro_rec, 3), 'Precision': round(macro_pre, 3),
#             'TrainF1': round(np.mean(tr_f1s), 3), 'Gap': round(gap, 3),
#             '과적합': '⚠️' if gap > 0.15 else '✅'}

In [32]:
def eval_line_model(sub, feats, model_name, model_builder, n_splits):
    X = sub[feats]
    yq = sub['Y_Quality'].values
    yc = sub['Y_Class'].values
    n = len(sub)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_pred = np.full(n, -1)
    tr_f1s, te_f1s = [], []

    for tr, te in skf.split(X, yc):
        # ── ① inner_cv(fit_idx/stop_idx) 분할 자체를 삭제, tr을 그대로 씀 ──
        # k_feat = max(5, min(len(feats), len(tr) // 5))
        k_feat = 60 # Top-N 60개 피쳐 선택

        # ── ② 피처선택·가중치·임계값·표준화 전부 tr 기준으로 되돌림 ──
        X_tr_imp = X.iloc[tr].fillna(X.iloc[tr].median()).fillna(0)
        mi = mutual_info_regression(X_tr_imp, yq[tr], random_state=42)
        top = X.columns[np.argsort(mi)[::-1][:k_feat]]
        Xtr, Xte = X.iloc[tr][top], X.iloc[te][top]

        w_tr = class_sample_weight(yc[tr])
        thr01, thr12 = fit_thresholds(yq[tr], yc[tr])
        mu, sd = yq[tr].mean(), yq[tr].std()
        sd = sd if sd > 1e-8 else 1.0
        yq_tr_s = (yq[tr] - mu) / sd

        # ── ③ eval_set / early_stopping 관련 인자를 fit()에서 전부 제거 ──
        model = model_builder()
        model.fit(Xtr, yq_tr_s, sample_weight=w_tr)   # 조기종료 없이 그냥 학습

        tr_pred_c = to_class(model.predict(Xtr) * sd + mu, thr01, thr12)
        te_pred_c = to_class(model.predict(Xte) * sd + mu, thr01, thr12)
        oof_pred[te] = te_pred_c
        tr_f1s.append(f1_score(yc[tr], tr_pred_c, average='macro'))
        te_f1s.append(f1_score(yc[te], te_pred_c, average='macro'))

    macro_f1 = f1_score(yc, oof_pred, average='macro')
    macro_rec = recall_score(yc, oof_pred, average='macro', zero_division=0)
    macro_pre = precision_score(yc, oof_pred, average='macro', zero_division=0)
    gap = np.mean(tr_f1s) - np.mean(te_f1s)

    return {'n': n, 'p_selected': k_feat, 'p/n': round(k_feat / len(tr), 2),
            'MacroF1': round(macro_f1, 3), 'Recall': round(macro_rec, 3), 'Precision': round(macro_pre, 3),
            'TrainF1': round(np.mean(tr_f1s), 3), 'Gap': round(gap, 3),
            '과적합': '⚠️' if gap > 0.15 else '✅'}

### 전체 실행

### 종합 성능</br>
- 단순 평균: 라인 하나하나를 동등하게 취급
- 가중 평균: 표본 많은 라인의 안정적 추정에 비중

In [33]:
if __name__ == '__main__':
    lines = load_data()
    models = {'LightGBM' :lambda: build_model()['LightGBM'],
              'XGBoost' :lambda: build_model()['XGBoost'],
              'CatBoost' :lambda: build_model()['CatBoost']}

    rows = []
    for ln, sub, feats in lines:
        min_class = pd.Series(sub['Y_Class']).value_counts().min()
        n_splits = 5 if min_class >= 15 else 3
        print(f'라인 {ln} - 샘플 수: {len(sub)}, 원본피처={len(feats)}, 클래스 최소 샘플 수: {min_class}, KFold 분할 수: {n_splits}')
        for mname in ['LightGBM', 'XGBoost', 'CatBoost']:
            res = eval_line_model(sub, feats, mname, models[mname], n_splits)
            res.update({'LINE':ln, 'MODEL':mname})
            rows.append(res)
            print(f'{mname:9s}: MacroF1={res["MacroF1"]:.3f}  Recall={res["Recall"]:.3f}  '
                  f'Precision={res["Precision"]:.3f}  gap={res["Gap"]:+.3f} {res["과적합"]}')

    result = pd.DataFrame(rows)[['LINE','MODEL','n','p_selected','p/n','MacroF1','Recall','Precision','TrainF1','Gap','과적합']]

    print('\n라인 x 모델 종합 결과\n')
    print(result.to_string(index=False))
    # result.to_csv('baseline_modeling_results_vanilla.csv', index=False)

    print('\n 라인별 최고 모델 (MacroF1 기준)')
    best = result.loc[result.groupby('LINE')['MacroF1'].idxmax()]
    print(best[['LINE','MODEL','MacroF1','Recall','Precision','Gap']].to_string(index=False))

    print('\n\n모델별 종합 성능 (6개 라인 통합)')
    summary_simple = result.groupby('MODEL')[['MacroF1','Recall','Precision','Gap']].mean().round(3)
    summary_simple['과적합_평균'] = summary_simple['Gap'].apply(lambda g: '⚠️' if g > 0.15 else '✅')

    def wavg(g, col):
        return np.average(g[col], weights=g['n'])
    summary_w = result.groupby('MODEL').apply(
        lambda g: pd.Series({
            'MacroF1_w': round(wavg(g,'MacroF1'),3),
            'Recall_w': round(wavg(g,'Recall'),3),
            'Precision_w': round(wavg(g,'Precision'),3),
            'Gap_w': round(wavg(g,'Gap'),3),
        })
    ).round(3)

    combined = summary_simple[['MacroF1','Recall','Precision','Gap']].join(summary_w)
    combined['과적합'] = combined['Gap_w'].apply(lambda g: '⚠️' if g > 0.15 else '✅')
    combined = combined.rename(columns={'MacroF1':'MacroF1_단순평균','Recall':'Recall_단순평균',
                                         'Precision':'Precision_단순평균','Gap':'Gap_단순평균'})
    print(combined.to_string())
    # combined.to_csv('model_summary_vanilla.csv')

    print('\n※ 단순평균 = 6개 라인을 동등 가중 / 가중평균(_w) = 라인 표본수(n)로 가중')
    # print('※ 라인별 세부 결과는 baseline_modeling_results_vanilla.csv 참고')

라인 T050304 - 샘플 수: 91, 원본피처=548, 클래스 최소 샘플 수: 11, KFold 분할 수: 3
LightGBM : MacroF1=0.443  Recall=0.440  Precision=0.446  gap=+0.409 ⚠️
XGBoost  : MacroF1=0.632  Recall=0.602  Precision=0.718  gap=+0.383 ⚠️
CatBoost : MacroF1=0.655  Recall=0.623  Precision=0.765  gap=+0.357 ⚠️
라인 T050307 - 샘플 수: 68, 원본피처=548, 클래스 최소 샘플 수: 12, KFold 분할 수: 3
LightGBM : MacroF1=0.524  Recall=0.515  Precision=0.644  gap=+0.326 ⚠️
XGBoost  : MacroF1=0.653  Recall=0.626  Precision=0.735  gap=+0.356 ⚠️
CatBoost : MacroF1=0.743  Recall=0.718  Precision=0.876  gap=+0.303 ⚠️
라인 T010306 - 샘플 수: 84, 원본피처=548, 클래스 최소 샘플 수: 8, KFold 분할 수: 3
LightGBM : MacroF1=0.561  Recall=0.561  Precision=0.567  gap=+0.230 ⚠️
XGBoost  : MacroF1=0.724  Recall=0.721  Precision=0.732  gap=+0.275 ⚠️
CatBoost : MacroF1=0.462  Recall=0.482  Precision=0.463  gap=+0.539 ⚠️
라인 T010305 - 샘플 수: 73, 원본피처=548, 클래스 최소 샘플 수: 16, KFold 분할 수: 5
LightGBM : MacroF1=0.607  Recall=0.596  Precision=0.625  gap=+0.173 ⚠️
XGBoost  : MacroF1=0.624  Recall=0.